## Реализация alignment через библиотеку TRL
После обсуждения теории рассмотрим обучение DPO на нескольких примерах с помощью библиотеки TRL.

### Задание 1
Приведите датасет к нужному формату. У вас есть примеры промпта, предпочтительного и отрицательного ответов. Составьте из них датасет, используя класс Dataset из библиотеки datasets так, чтобы он содержал два поля сообщений для обучения модели: 
- chosen в качестве ответа модели содержит пример chosen
- rejected в качестве ответа модели содержит значение из примера rejected.

Используйте токенизатор от модели `Qwen/Qwen2.5-0.5B-Instruct`.
Пояснение: на вход токенизатора идёт список сообщений в формате словарей с двумя ключами: 'role' и 'content'.
- Для поля chosen сформируйте список из двух сообщений: входной промпт (значение 'content' при 'role': 'user') и выбранный правильный ответ (значение'content' при 'role': 'assistant').
- Для поля rejected сформируйте аналогичный список: тот же промпт (значение 'content' при 'role': 'user') и неправильный ответ (значение 'content' при 'role': 'assistant').

In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

examples = [
    {
        "prompt": "Объясни, почему небо голубое.",
        "chosen": "Потому что молекулы воздуха рассеивают короткие волны света сильнее длинных, поэтому мы видим преимущественно голубую часть спектра.",
        "rejected": "Потому что так захотела природа, и это просто красиво.",
    },
    {
        "prompt": "Дай безопасный совет по хранению паролей.",
        "chosen": "Используйте менеджер паролей и включите двухфакторную аутентификацию; не повторяйте один и тот же пароль на разных сайтах.",
        "rejected": "Запишите все пароли в заметке на телефоне, так их легче не забыть.",
    },
]

def examples_to_messages(examples):
    data = {'chosen': [], 'rejected': []}
    for example in examples:
        data['chosen'].append([
            {'role': 'user', 'content': example['prompt']},
            {'role': 'assistant', 'content': example['chosen']}
        ])
        data['rejected'].append([
            {'role': 'user', 'content': example['prompt']},
            {'role': 'assistant', 'content': example['rejected']}
        ])
    return Dataset.from_dict(data)

ds = examples_to_messages(examples)
ds

/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 6/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['chosen', 'rejected'],
    num_rows: 2
})

Для дальнейшего обучения достаточно только этого датасета — библиотека TRL сама предобработает данные чатом-шаблоном и токенизирует их.
Рассмотрим пример обучения модели методом DPO: 

1. Импортируем класс `DPOTrainer` — цикл обучения и `DPOConfig` — гиперпараметры обучения
1. Инициализируем модели из Qwen-модели, которая уже прошла этапы предобучения и SFT

In [2]:
from trl import DPOTrainer, DPOConfig

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
# эту модель мы будем обучать
model = AutoModelForCausalLM.from_pretrained(model_id)
# эту модель будем использовать для сравнения распределения вероятностей
ref_model = AutoModelForCausalLM.from_pretrained(model_id)
# заморозим её веса
ref_model.eval()

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 11449.71it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

3. Зададим гиперпараметры:

In [3]:
config = DPOConfig(
    beta=0.1, # параметр beta              
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    max_length=512,
    # max_prompt_length=256,
    num_train_epochs=3,
    report_to='none',
    logging_steps=1,
    save_strategy='no'
)
tokenizer.model_max_length = 256

4. Запустим обучение модели на наших данных с учётом гиперпараметров:

In [5]:
trainer = DPOTrainer(
    model,
    ref_model,
    args=config,
    train_dataset=ds,
    processing_class=tokenizer,
)

trainer.train()

Tokenizing train dataset: 100%|██████████| 2/2 [00:00<00:00, 95.05 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 6/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.694683
2,0.618877
3,0.002010
4,0.017427
5,0.000315
6,0.005636


TrainOutput(global_step=6, training_loss=0.22315780252877934, metrics={'train_runtime': 31.8364, 'train_samples_per_second': 0.188, 'train_steps_per_second': 0.188, 'total_flos': 2216105109504.0, 'train_loss': 0.22315780252877934})